# CO2 injection cost inputs

This notebook replaces the injection-cost spreadsheet. It calculates the same basin-level investment and operating costs, then writes those values into the local `assets/assets_1/co2_injection.csv`.

Run order is intentionally simple: imports and paths, assumptions, formulas, CSV update, validation.

In [ ]:
from pathlib import Path
import re

import pandas as pd

In [ ]:
def find_project_root(start=None):
    """Return the nearest parent directory that contains the model asset folder."""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "assets" / "assets_1" / "co2_injection.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find assets/assets_1/co2_injection.csv")


PROJECT_ROOT = find_project_root()
CO2_INJECTION_PATH = PROJECT_ROOT / "assets" / "assets_1" / "co2_injection.csv"
CO2_INJECTION_PATH

## Assumptions

The constants below match the original spreadsheet. Change them here when the cost assumptions change.

In [ ]:
COST_ASSUMPTIONS = {
    "Cdr ($/m)": 28275.0,
    "Cfx ($)": 8917500.0,
    "Cfc ($)": 6655500.0,
    "Csd ($)": 26205487.5,
    "Cme ($)": 1663875.0,
    "Cac (%)": 0.05,
}

HOURS_PER_YEAR = 8760
TONS_PER_MEGATON = 1_000_000
OPERATIONAL_COST_FRACTION = 0.03

basin_inputs = pd.DataFrame([{'Basin': 'Songliao Basin', 'Median Depth (m)': 1500, 'Injection Rate (Mt/a)': 0.42, 'Injection Rate (Mt/a) Max': 1.64}, {'Basin': 'Turpan-Hami Basin', 'Median Depth (m)': 3000, 'Injection Rate (Mt/a)': 10.26, 'Injection Rate (Mt/a) Max': 28.63}, {'Basin': 'Subei Basin', 'Median Depth (m)': 1750, 'Injection Rate (Mt/a)': 13.58, 'Injection Rate (Mt/a) Max': 34.07}, {'Basin': 'Bohai Bay Basin (onshore)', 'Median Depth (m)': 2250, 'Injection Rate (Mt/a)': 23.82, 'Injection Rate (Mt/a) Max': 122.77}, {'Basin': 'Qaidam Basin', 'Median Depth (m)': 3000, 'Injection Rate (Mt/a)': 9.43, 'Injection Rate (Mt/a) Max': 17.56}, {'Basin': 'Nanxiang Basin', 'Median Depth (m)': 2250, 'Injection Rate (Mt/a)': 14.86, 'Injection Rate (Mt/a) Max': 124.67}, {'Basin': 'Sanjiang Basin', 'Median Depth (m)': 2000, 'Injection Rate (Mt/a)': 50.05, 'Injection Rate (Mt/a) Max': 94.65}, {'Basin': 'Hailar Basin', 'Median Depth (m)': 1750, 'Injection Rate (Mt/a)': 0.02, 'Injection Rate (Mt/a) Max': 0.06}, {'Basin': 'Jianghan Basin', 'Median Depth (m)': 2000, 'Injection Rate (Mt/a)': 0.08, 'Injection Rate (Mt/a) Max': 0.31}, {'Basin': 'Tarim Basin', 'Median Depth (m)': 2000, 'Injection Rate (Mt/a)': 5.78, 'Injection Rate (Mt/a) Max': 32.5}, {'Basin': 'Ordos Basin', 'Median Depth (m)': 2250, 'Injection Rate (Mt/a)': 0.1, 'Injection Rate (Mt/a) Max': 0.79}, {'Basin': 'Ejinjina Basin', 'Median Depth (m)': 1500, 'Injection Rate (Mt/a)': 0.05, 'Injection Rate (Mt/a) Max': 0.12}, {'Basin': 'Hehuai Basin', 'Median Depth (m)': 2500, 'Injection Rate (Mt/a)': 3.58, 'Injection Rate (Mt/a) Max': 51.87}, {'Basin': 'Qinshui Basin', 'Median Depth (m)': 1100, 'Injection Rate (Mt/a)': 0.12, 'Injection Rate (Mt/a) Max': 0.17}, {'Basin': 'Erlian Basin', 'Median Depth (m)': 900, 'Injection Rate (Mt/a)': 4.56, 'Injection Rate (Mt/a) Max': 14.67}, {'Basin': 'Junggar Basin', 'Median Depth (m)': 2500, 'Injection Rate (Mt/a)': 0.8, 'Injection Rate (Mt/a) Max': 2.28}, {'Basin': 'Sichuan Basin', 'Median Depth (m)': 1900, 'Injection Rate (Mt/a)': 0.01, 'Injection Rate (Mt/a) Max': 0.02}, {'Basin': 'Bohai Bay Basin (offshore)', 'Median Depth (m)': 1450, 'Injection Rate (Mt/a)': 16.81, 'Injection Rate (Mt/a) Max': 34.44}, {'Basin': 'North Yellow Sea Basin', 'Median Depth (m)': 1500, 'Injection Rate (Mt/a)': 14.77, 'Injection Rate (Mt/a) Max': 67.35}, {'Basin': 'South Yellow Sea Basin', 'Median Depth (m)': 2500, 'Injection Rate (Mt/a)': 0.07, 'Injection Rate (Mt/a) Max': 0.61}, {'Basin': 'East China Sea Basin', 'Median Depth (m)': 1600, 'Injection Rate (Mt/a)': 12.0, 'Injection Rate (Mt/a) Max': 61.65}, {'Basin': 'Pearl River Mouth Basin', 'Median Depth (m)': 1500, 'Injection Rate (Mt/a)': 9.3, 'Injection Rate (Mt/a) Max': 104.24}, {'Basin': 'Beibu Gulf Basin', 'Median Depth (m)': 1500, 'Injection Rate (Mt/a)': 24.92, 'Injection Rate (Mt/a) Max': 101.3}, {'Basin': 'Qiongdongnan Basin', 'Median Depth (m)': 2250, 'Injection Rate (Mt/a)': 44.43, 'Injection Rate (Mt/a) Max': 134.4}])
basin_inputs

## Formulas

These formulas mirror the spreadsheet:

- `Injection Rate (t/hr) Mean = Injection Rate (Mt/a) * 1e6 / 8760`
- `Injection Rate (t/hr) Max = Injection Rate (Mt/a) Max * 1e6 / 8760`
- `Costj(WellFixed) = (Cdr * Median Depth + Cfx + Cfc) * (1 + Cac)`
- `Costj(ResFixed) = (Csd + Cme) * (1 + Cac)`
- `Cj(WF) + Cj(RF) = Costj(WellFixed) + Costj(ResFixed)`
- `Unit Cost of Injection ($/(t/hr)) - investment cost = total fixed cost / mean injection rate`
- `Operational cost = investment cost * 0.03`

In [ ]:
INVESTMENT_COST_COL = "Unit Cost of Injection ($/(t/hr)) - investment cost"
OPERATIONAL_COST_COL = "Operational cost"


def calculate_injection_costs(basin_inputs, assumptions):
    costs = basin_inputs.copy()

    costs["Injection Rate (t/hr) Mean"] = (
        costs["Injection Rate (Mt/a)"] * TONS_PER_MEGATON / HOURS_PER_YEAR
    )
    costs["Injection Rate (t/hr) Max"] = (
        costs["Injection Rate (Mt/a) Max"] * TONS_PER_MEGATON / HOURS_PER_YEAR
    )

    for column, value in assumptions.items():
        costs[column] = value

    costs["Costj(WellFixed) ($)"] = (
        costs["Cdr ($/m)"] * costs["Median Depth (m)"]
        + costs["Cfx ($)"]
        + costs["Cfc ($)"]
    ) * (1 + costs["Cac (%)"])
    costs["Costj(ResFixed) ($)"] = (
        costs["Csd ($)"] + costs["Cme ($)"]
    ) * (1 + costs["Cac (%)"])
    costs["Cj(WF) + Cj(RF) ($)"] = (
        costs["Costj(WellFixed) ($)"] + costs["Costj(ResFixed) ($)"]
    )

    costs[INVESTMENT_COST_COL] = (
        costs["Cj(WF) + Cj(RF) ($)"] / costs["Injection Rate (t/hr) Mean"]
    )
    costs["Unit Cost of Injection ($/(Mt/year)) Mean"] = (
        costs["Cj(WF) + Cj(RF) ($)"] / costs["Injection Rate (Mt/a)"]
    )
    costs["Unit Cost of Injection ($/(Mt/year)) Max"] = (
        costs["Cj(WF) + Cj(RF) ($)"] / costs["Injection Rate (Mt/a) Max"]
    )
    costs["Mean to Max Unit Cost Ratio"] = (
        costs["Unit Cost of Injection ($/(Mt/year)) Mean"]
        / costs["Unit Cost of Injection ($/(Mt/year)) Max"]
    )
    costs[OPERATIONAL_COST_COL] = costs[INVESTMENT_COST_COL] * OPERATIONAL_COST_FRACTION
    return costs


costs_df = calculate_injection_costs(basin_inputs, COST_ASSUMPTIONS)
injection_costs = costs_df[["Basin", INVESTMENT_COST_COL, OPERATIONAL_COST_COL]].copy()
injection_costs

## Update `co2_injection.csv`

The CSV has one row per source-to-storage link. The cost table has one row per storage basin. Matching therefore uses the storage vertex column, `edges--co2_storage_edge--end_vertex`.

In [ ]:
INJECTION_STORAGE_VERTEX_COL = "edges--co2_storage_edge--end_vertex"
INJECTION_INVESTMENT_TARGET_COL = "edges--co2_captured_edge--investment_cost"
INJECTION_OPERATIONAL_TARGET_COL = "edges--co2_captured_edge--variable_om_cost"

BASIN_NAME_ALIASES = {
    "Bohai Bay Basin (onshore)": "BohaiOnshore",
    "Bohai Bay Basin (offshore)": "BohaiOffshore",
    "Sichuan Basin": "SichuanBasin",
    "Turpan-Hami Basin": "TurpanHami",
    "Yingen-Ejina Basin": "YingenEjina",
    "Ejinjina Basin": "YingenEjina",
}


def clean_node_name(value):
    return re.sub(r"[^A-Za-z0-9]", "", str(value)).lower()


def basin_key(value):
    name = BASIN_NAME_ALIASES.get(str(value), str(value))
    if name.endswith(" Basin"):
        name = name.removesuffix(" Basin")
    return clean_node_name(name)


def storage_vertex_key(value):
    return clean_node_name(str(value).removeprefix("co2_storage_"))


def require_columns(df, required_columns, table_name):
    missing_columns = set(required_columns) - set(df.columns)
    if missing_columns:
        raise ValueError(f"{table_name} is missing columns: {sorted(missing_columns)}")


def build_cost_lookup(injection_costs):
    require_columns(injection_costs, {"Basin", INVESTMENT_COST_COL, OPERATIONAL_COST_COL}, "injection_costs")
    lookup = injection_costs.assign(
        _basin_key=injection_costs["Basin"].map(basin_key),
    )[["_basin_key", "Basin", INVESTMENT_COST_COL, OPERATIONAL_COST_COL]]

    duplicated = lookup.duplicated("_basin_key", keep=False)
    if duplicated.any():
        raise ValueError(
            "Duplicate basin keys found:\n"
            + lookup.loc[duplicated, ["Basin", "_basin_key"]].to_string(index=False)
        )
    return lookup


def update_injection_csv(csv_path, injection_costs):
    injection = pd.read_csv(csv_path, dtype=str)
    require_columns(
        injection,
        {
            "id",
            INJECTION_STORAGE_VERTEX_COL,
            INJECTION_INVESTMENT_TARGET_COL,
            INJECTION_OPERATIONAL_TARGET_COL,
        },
        str(csv_path),
    )

    matched = injection.assign(
        _basin_key=injection[INJECTION_STORAGE_VERTEX_COL].map(storage_vertex_key),
    ).merge(
        build_cost_lookup(injection_costs),
        on="_basin_key",
        how="left",
        validate="many_to_one",
        indicator=True,
    )

    unmatched = matched[matched["_merge"] != "both"]
    if not unmatched.empty:
        raise ValueError(
            "Some CO2 injection rows could not be matched to basin costs:\n"
            + unmatched[["id", INJECTION_STORAGE_VERTEX_COL]].to_string(index=False)
        )

    matched[INJECTION_INVESTMENT_TARGET_COL] = matched[INVESTMENT_COST_COL]
    matched[INJECTION_OPERATIONAL_TARGET_COL] = matched[OPERATIONAL_COST_COL]
    matched[injection.columns].to_csv(csv_path, index=False)
    return len(matched)


rows_updated = update_injection_csv(CO2_INJECTION_PATH, injection_costs)
print(f"Updated {rows_updated} rows in {CO2_INJECTION_PATH}")

In [ ]:
def validate_written_costs(csv_path, injection_costs):
    written = pd.read_csv(csv_path)
    checked = written.assign(
        _basin_key=written[INJECTION_STORAGE_VERTEX_COL].map(storage_vertex_key),
    ).merge(
        build_cost_lookup(injection_costs),
        on="_basin_key",
        how="left",
        validate="many_to_one",
    )

    return {
        "path": str(csv_path),
        "rows_checked": len(checked),
        "investment_costs_match": bool(
            checked[INJECTION_INVESTMENT_TARGET_COL].round(10).eq(
                checked[INVESTMENT_COST_COL].round(10)
            ).all()
        ),
        "operational_costs_match": bool(
            checked[INJECTION_OPERATIONAL_TARGET_COL].round(10).eq(
                checked[OPERATIONAL_COST_COL].round(10)
            ).all()
        ),
    }


pd.DataFrame([validate_written_costs(CO2_INJECTION_PATH, injection_costs)])